# Chapter 14 — Poisson Processes in Systems
*Track 4: Engineers*

## 🎯 Learning Objectives
- Model random event arrivals with the Poisson process
- Derive inter-arrival times and their exponential distribution
- Apply to server requests, hardware failures, and network packets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — The Poisson Process

A **Poisson process** with rate $\lambda$ satisfies:
1. **Independence**: events in non-overlapping intervals are independent
2. **Stationarity**: rate is constant over time
3. **Singularity**: no two events at the same instant

**Key results:**
- Number of events in time $t$: $N(t) \sim \text{Poisson}(\lambda t)$
- Inter-arrival time: $T \sim \text{Exp}(\lambda)$ (memoryless!)
- Merging two Poisson processes ($\lambda_1$, $\lambda_2$) → $\text{Poisson}(\lambda_1 + \lambda_2)$

In [ ]:
# Simulate a Poisson process
lambda_rate = 3  # events per unit time
T_max = 20

inter_arrivals = rng.exponential(1/lambda_rate, 200)
arrival_times = np.cumsum(inter_arrivals)
arrival_times = arrival_times[arrival_times <= T_max]

# Count process N(t)
t_grid = np.linspace(0, T_max, 1000)
N_t = np.array([np.sum(arrival_times <= t) for t in t_grid])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].step(t_grid, N_t, where="post")
axes[0].set_xlabel("Time"); axes[0].set_ylabel("N(t)")
axes[0].set_title(f"Poisson Process (λ={lambda_rate})")

axes[1].hist(inter_arrivals[:50], bins=20, density=True, alpha=0.6)
t_range = np.linspace(0, 2, 200)
axes[1].plot(t_range, stats.expon.pdf(t_range, scale=1/lambda_rate), "r-", lw=2,
             label=f"Exp(λ={lambda_rate})")
axes[1].set_title("Inter-arrival Times"); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"Expected arrivals in T={T_max}: {lambda_rate*T_max:.0f}")
print(f"Simulated arrivals: {len(arrival_times)}")

## 2. Math Walkthrough — Poisson PMF Derivation

Starting from the binomial: divide $[0,t]$ into $n$ tiny intervals of size $\Delta t = t/n$.

$$P(N(t) = k) = \binom{n}{k}(\lambda\Delta t)^k (1-\lambda\Delta t)^{n-k}$$

As $n \to \infty$:
$$P(N(t) = k) = \frac{(\lambda t)^k e^{-\lambda t}}{k!}$$

In [ ]:
# Verify: Poisson counts in windows
lambda_rate = 5
window_size = 1.0
n_windows = 5000
counts = rng.poisson(lambda_rate * window_size, n_windows)

k_range = range(0, 15)
empirical = [np.mean(counts == k) for k in k_range]
theoretical = [stats.poisson.pmf(k, lambda_rate) for k in k_range]

plt.bar([k-0.2 for k in k_range], empirical, width=0.4, alpha=0.7, label="Simulated")
plt.bar([k+0.2 for k in k_range], theoretical, width=0.4, alpha=0.7, label="Theoretical")
plt.xlabel("Count"); plt.ylabel("Probability")
plt.title(f"Poisson(λ={lambda_rate}) — Empirical vs Theoretical")
plt.legend(); plt.tight_layout(); plt.show()

## 3–6. Applications in Engineering

In [ ]:
# Server failure events — non-homogeneous Poisson process (time-varying rate)
T_hours = 24
t_fine = np.linspace(0, T_hours, 1000)

# Rate varies with time of day (peak at 9am and 9pm)
def lambda_t(t):
    return 2 + 5*np.exp(-0.5*(t-9)**2/4) + 3*np.exp(-0.5*(t-21)**2/4)

plt.figure(figsize=(10, 4))
plt.plot(t_fine, lambda_t(t_fine), lw=2)
plt.fill_between(t_fine, lambda_t(t_fine), alpha=0.3)
plt.xlabel("Hour of day"); plt.ylabel("Event rate λ(t)")
plt.title("Non-Homogeneous Poisson Process — Server Events Over 24 Hours")
plt.tight_layout(); plt.show()

# Total expected events
from scipy.integrate import quad
total_expected, _ = quad(lambda_t, 0, T_hours)
print(f"Total expected events in 24h: {total_expected:.0f}")

In [ ]:
# Network packet arrivals — test if Poisson assumption holds
packet_gaps = rng.exponential(0.001, 2000)  # 1ms mean inter-arrival (1000 pps)

# Test: Poisson requires exponential inter-arrivals
D_stat, p_val = stats.kstest(packet_gaps, "expon",
                              args=(packet_gaps.min(), packet_gaps.mean() - packet_gaps.min()))
print(f"KS test for exponential inter-arrivals: D={D_stat:.4f}, p={p_val:.4f}")
print("Poisson assumption ✅ plausible" if p_val > 0.05 else "Poisson assumption ❌ rejected")

# Compound Poisson: burst packets (each arrival brings a batch)
batch_sizes = rng.geometric(0.3, 500)
total_packets = batch_sizes.sum()
print(f"
Burst traffic: {len(batch_sizes)} arrivals, {total_packets} total packets")
print(f"Mean batch size: {batch_sizes.mean():.2f}")

In [ ]:
# Practice
lambda_sys = 0.5  # failures per day
# P(zero failures in a week)
p_zero = stats.poisson.pmf(0, lambda_sys*7)
print(f"P(0 failures in 7 days): {p_zero:.4f}")
# Expected wait for 3rd failure
from scipy.stats import erlang
mttf_3rd = 3 / lambda_sys
print(f"Expected time to 3rd failure: {mttf_3rd:.1f} days")
# Inter-arrival 90th percentile
p90 = stats.expon.ppf(0.90, scale=1/lambda_sys)
print(f"90th percentile inter-arrival time: {p90:.2f} days")